In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/after_encoding/df_encoded.csv")

In [3]:
import pandas as pd
import numpy as np

print("🔧 GENERATING NEW TARGETS...")
print("="*60)

# ==========================================
# TARGET 4: TIME TO TRACTION (time_to_1000_users_months)
# ==========================================
print("\n📊 Target 4: Time to 1000 Users")

# Check if column exists and has data
if 'time_to_1000_users_months' in df.columns:
    print(f"   Original column found!")
    print(f"   Non-null values: {df['time_to_1000_users_months'].notna().sum()}")
    print(f"   Range: {df['time_to_1000_users_months'].min():.1f} - {df['time_to_1000_users_months'].max():.1f} months")
    
    # Clean the target
    df['target_traction_time'] = df['time_to_1000_users_months'].clip(lower=0, upper=60)  # Cap at 5 years
    
else:
    print("   Column not found! Creating synthetic estimate...")
    # Estimate based on downloads growth
    df['target_traction_time'] = np.where(
        df['downloads'] >= 1000,
        np.maximum(1, df['age_months'] * 0.3),  # Estimate: reached 1k users in ~30% of lifetime
        np.nan
    )

print(f"   Final target stats:")
print(f"   Valid samples: {df['target_traction_time'].notna().sum()}")
print(f"   Mean: {df['target_traction_time'].mean():.1f} months")
print(f"   Median: {df['target_traction_time'].median():.1f} months")


# ==========================================
# TARGET 5: BREAK-EVEN TIME (break_even_time_months)
# ==========================================
print("\n📊 Target 5: Break-Even Time")

if 'break_even_time_months' in df.columns:
    print(f"   Original column found!")
    print(f"   Non-null values: {df['break_even_time_months'].notna().sum()}")
    print(f"   Range: {df['break_even_time_months'].min():.1f} - {df['break_even_time_months'].max():.1f} months")
    
    # Clean the target
    df['target_breakeven_time'] = df['break_even_time_months'].clip(lower=0, upper=84)  # Cap at 7 years
    
else:
    print("   Column not found! Creating estimate from profitability...")
    # Estimate: if profitable now, when did they break even?
    # If not profitable, estimate based on burn/revenue ratio
    df['target_breakeven_time'] = np.where(
        df['is_profitable'] == 1,
        df['age_months'] * 0.6,  # Assume broke even at 60% of current age
        np.nan  # Can't estimate for non-profitable
    )

print(f"   Final target stats:")
print(f"   Valid samples: {df['target_breakeven_time'].notna().sum()}")
print(f"   Mean: {df['target_breakeven_time'].mean():.1f} months")
print(f"   Median: {df['target_breakeven_time'].median():.1f} months")

print("\n✅ TARGETS GENERATED!")
print("="*60)


🔧 GENERATING NEW TARGETS...

📊 Target 4: Time to 1000 Users
   Column not found! Creating synthetic estimate...
   Final target stats:
   Valid samples: 44401
   Mean: 18.8 months
   Median: 14.3 months

📊 Target 5: Break-Even Time
   Column not found! Creating estimate from profitability...
   Final target stats:
   Valid samples: 701
   Mean: 34.0 months
   Median: 23.8 months

✅ TARGETS GENERATED!


In [13]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

print("="*60)
print("📊 MODEL 4: TIME TO TRACTION")
print("="*60)

# ==========================================
# DEFINE LEAKS
# ==========================================
COMMON_LEAKS = [
    'success_label', 'still_operating', 'is_dead',
    'exit_status_Active', 'exit_status_Bankruptcy', 'exit_status_Failed',
    'exit_status_Operating', 'exit_status_Shut Down', 'exit_status_Shutdown',
    'exit_status_Still Active', 'exit_status_Unknown', 'exit_status_unknown',
    'failure_reason_cat_business_model', 'failure_reason_cat_competition',
    'failure_reason_cat_funding', 'failure_reason_cat_legal',
    'failure_reason_cat_market_fit', 'failure_reason_cat_multiple',
    'failure_reason_cat_no_market_fit', 'failure_reason_cat_other',
    'failure_reason_cat_ran_out_of_cash', 'failure_reason_cat_team',
    'failure_reason_cat_team_issues', 'failure_reason_cat_unknown',
    'failure_reason_cat_encoded',
    'success_reason_cat_revenue', 'success_reason_cat_unknown',
    'synthetic_data', 'data_source_synthetic', 'data_source_synthetic_augmentation'
]

TEXT_LEAKS = [c for c in df.columns if c.startswith('sf_') or c.startswith('fr_') or c.startswith('sr_')]

TRACTION_LEAKS = COMMON_LEAKS + TEXT_LEAKS + [
    # Target
    'time_to_1000_users_months',
    'target_traction_time',
    
    # Traction metrics (can't use current traction to predict time TO traction)
    'downloads', 'downloads_per_month',
    'active_users',
    'upvotes', 'upvotes_per_month',
    'review_count', 'reviews_per_month',
    'engagement_score',
    'has_viral_reach',
    'rating_avg', 'rating_weighted', 'rating_confidence',
    'rating_tier', 'is_highly_rated', 'is_top_rated',
    'has_strong_reviews', 'has_strong_rating',
    'rating_vs_category',
    
    # Age/time (circular with target)
    'age_days', 'age_months', 'age_years',
    'is_young', 'is_established', 'is_old',
    'survival_months',
    
    # Revenue/financial (comes after traction)
    'revenue_monthly', 'revenue_monthly_filled', 'revenue_estimated',
    'monthly_profit_est', 'is_profitable',
    'burn_rate_monthly_est',
    'estimated_runway_months',
    
    # Derived scores
    'survival_score', 'growth_score', 'market_fit_score', 'risk_score',
    'capital_efficiency', 'revenue_to_burn_ratio',
    
    # Other model outputs
    'target_survival_months', 'target_growth_rate', 'target_growth_rate_log',
    'target_raise_funding', 'market_value_estimate', 'target_breakeven_time',
    
    # Metadata flags
    'has_marketing_channels', 'has_business_model', 'has_case_study_url',
    'has_brand', 'has_founders', 'has_website_url',
    'info_completeness', 'has_long_description', 'has_detailed_name',
    
    # Data source bias
    'data_source_failory', 'data_source_producthunt', 'data_source_indiehackers',
    'data_source_playstore', 'source_encoded',
    
    # Adjustments/interactions
    'rating_adjustment', 'team_adjustment', 'funding_adjustment',
    'funding_team_interaction', 'quality_traction_interaction',
    'maturity_quality_interaction', 'price_reach_interaction',
    
    # Unknown flags
    'business_model_unknown', 'product_type_unknown', 'main_category_unknown'
]

# ==========================================
# PREPARE DATA
# ==========================================
print("\n📊 Preparing data...")
X_trac = df.drop(columns=TRACTION_LEAKS, errors='ignore')
y_trac = df['target_traction_time']

# Remove missing
mask = y_trac.notna()
X_trac = X_trac[mask]
y_trac = y_trac[mask]

print(f"   Samples: {len(X_trac)}")
print(f"   Features: {X_trac.shape[1]}")
print(f"   Target range: {y_trac.min():.1f} - {y_trac.max():.1f} months")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_trac, y_trac, test_size=0.2, random_state=42
)

# ==========================================
# TRAIN MODEL
# ==========================================
print("\n🥊 Training XGBoost...")
trac_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.035,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

trac_model.fit(X_train, y_train)
print("✅ Training complete!")

# ==========================================
# EVALUATE
# ==========================================
y_pred = trac_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n🏆 TIME TO TRACTION RESULTS:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE:      {mae:.2f} months")
print(f"   RMSE:     {rmse:.2f} months")

# Feature importance
imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': trac_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(f"\n🔍 TOP 15 FEATURES:")
for i, row in imp.head(15).iterrows():
    print(f"   {row['Feature']:<35} {row['Importance']:.4f}")

# Decision
if r2 > 0.70:
    print(f"\n⚠️ R²={r2:.4f} - check for leaks")
elif r2 > 0.30:
    print(f"\n✅ GOOD! R²={r2:.4f} is realistic!")
    joblib.dump(trac_model, '../models/traction_time_model_v1.pkl')
    print("✅ Saved: traction_time_model_v1.pkl")
else:
    print(f"\n⚠️ R²={r2:.4f} is low")
    joblib.dump(trac_model, '../models/traction_time_model_v1.pkl')
    print("✅ Saved anyway")

print("\n" + "="*60)


📊 MODEL 4: TIME TO TRACTION

📊 Preparing data...
   Samples: 44401
   Features: 191
   Target range: 1.0 - 61.8 months

🥊 Training XGBoost...
✅ Training complete!

🏆 TIME TO TRACTION RESULTS:
   R² Score: 0.5894
   MAE:      7.21 months
   RMSE:     10.21 months

🔍 TOP 15 FEATURES:
   market_saturation                   0.3141
   product_type_app                    0.0959
   is_premium                          0.0505
   category_encoded                    0.0339
   desc_5                              0.0234
   product_type_digital_app            0.0153
   desc_56                             0.0111
   is_free                             0.0092
   price_vs_category                   0.0088
   desc_10                             0.0076
   desc_84                             0.0075
   is_well_funded                      0.0075
   desc_39                             0.0072
   main_category_travel                0.0069
   product_type_encoded                0.0067

✅ GOOD! R²=0.5894 is reali

In [18]:
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# ==========================================
# TRAIN STACKED MODEL
# ==========================================
print("\n🥞 Training STACKED Model (XGB + CAT → Ridge)...")

# Base 1: XGBoost
xgb_base = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Base 2: CatBoost (strong recommended config)
cat_base = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    bagging_temperature=1,
    random_state=42,
    verbose=False
)

# Meta learner: Ridge
ridge_meta = Ridge(alpha=1.0)

# Stacker
trac_stack = StackingRegressor(
    estimators=[
        ('xgb', xgb_base),
        ('cat', cat_base)
    ],
    final_estimator=ridge_meta,
    passthrough=True,   # gives meta the original features + model preds
    cv=5,
    n_jobs=-1
)

# Train the whole stack
trac_stack.fit(X_train, y_train)
print("✅ Stacking training complete!")

# ==========================================
# EVALUATE
# ==========================================
y_pred = trac_stack.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\n🏆 STACKED TIME TO TRACTION RESULTS:")
print(f"   R² Score: {r2:.4f}")
print(f"   MAE:      {mae:.2f} months")
print(f"   RMSE:     {rmse:.2f} months")

# Save model




🥞 Training STACKED Model (XGB + CAT → Ridge)...


C:\Users\mahes\anaconda3\Lib\site-packages\scipy\_lib\_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.26074e-27): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


✅ Stacking training complete!

🏆 STACKED TIME TO TRACTION RESULTS:
   R² Score: 0.6097
   MAE:      6.93 months
   RMSE:     9.95 months


In [20]:
joblib.dump(trac_stack, '../models/traction_time_model/tt_v1/traction_time_best_stack_model.pkl')
joblib.dump(xgb_base, '../models/traction_time_model/tt_v1/survival_month_xgb_base.pkl')
joblib.dump(cat_base, '../models/traction_time_model/tt_v1/survival_month_cat_base.pkl')
joblib.dump(X_train, '../models/traction_time_model/tt_v1/feature_names.pkl')
print("Saved!")

Saved!
